EXTRACTION ET ANALYSE DE LA CAPITALISATION DES CRYPTOMONNAIES

Dans ce projet j'ai construit un pipeline ETL ( Extraction, Transformation, Chargement) pour analyser l'évolution de la capitalisation boursière des cryptomonnais. Ce processus impliquera l'extraction de données à partir de source disponible sur internet, la transformation de ces données pour nettoyer et structurer l'information puis le chargement des résultats dans une base de données ou un fichier.

OBJECTIF

L'objectif de ce projet est d'extraire des données de capitalisation de cryptomonnaies sur plusieurs mois/années et d'analyser l'évolution de la capitalisation boursière en fonction du volume des trasactions. Et identifier aussi des pics ou des creux dans ces valeurs pour comprendre les périodes de forte volatilités.

In [ ]:
# Installation des packages
#! pip install polars
#! pip install pandas
# ! pip install pyspark 

In [17]:
# Importer les packages
import requests
import polars as pl
import pandas as pd
from pyspark.sql import SparkSession

In [ ]:

url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
        'vs_currency': 'usd',
        'order': 'market_cap_desc',
        'per_page': 1000, # Obtenir les 1000 premières cryptomonnaies de la capitalisation
        'page': 1,
        'sparline': 'false'
    }

# Requéte de l'API
response = requests.get(url, params = params)

# Verifier le statut si le code fonctionne
if response.status_code == 200: # C'est à dire que la requéte à fonctionner
    data = response.json()
    df = pd.json_normalize(data) # Convertir le json en Dataframe
else:
    raise Exception("Erreur lors de l'extraction des données de l'API")


In [37]:
# Afficher la base
df

,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated,roi.times,roi.currency,roi.percentage
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,106874.000000,2137642135270,1,2137642135270,6.826248e+10,110665.000000,...,-14.93356,2025-10-06T18:57:42.558Z,67.810000,1.580673e+05,2013-07-06T00:00:00.000Z,NaN,2025-11-03T18:46:01.818Z,NaN,NaN,NaN
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,3638.920000,440282466409,2,440282466409,4.346637e+10,3911.960000,...,-26.21922,2025-08-24T19:21:03.333Z,0.432979,8.427202e+05,2015-10-20T00:00:00.000Z,NaN,2025-11-03T18:46:04.089Z,44.533056,btc,4453.30564
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,1.000000,183464569414,3,183464569414,1.305912e+11,1.000000,...,-24.41789,2018-07-24T00:00:00.000Z,0.572521,7.467015e+01,2015-03-02T00:00:00.000Z,NaN,2025-11-03T18:45:58.444Z,NaN,NaN,NaN
3,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,2.360000,141998078888,4,236207775813,5.047472e+09,2.530000,...,-35.21562,2025-07-18T03:40:53.808Z,0.002686,8.784123e+04,2014-05-22T00:00:00.000Z,NaN,2025-11-03T18:45:58.194Z,NaN,NaN,NaN
4,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,997.650000,137485891151,5,137485890183,3.572191e+09,1086.670000,...,-27.12382,2025-10-13T08:41:24.131Z,0.039818,2.507322e+06,2017-10-19T00:00:00.000Z,NaN,2025-11-03T18:46:01.644Z,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,sky,sky,Sky,https://coin-images.coingecko.com/coins/images...,0.054403,1247575889,96,1279439536,1.558553e+07,0.056126,...,-45.65981,2024-12-04T10:13:57.264Z,0.035827,5.248450e+01,2025-02-03T05:01:48.626Z,NaN,2025-11-03T18:45:58.990Z,NaN,NaN,NaN
96,jupiter-exchange-solana,jup,Jupiter,https://coin-images.coingecko.com/coins/images...,0.369772,1191162508,97,2590172229,4.804983e+07,0.405421,...,-81.50778,2024-01-31T15:02:47.304Z,0.212468,7.407136e+01,2025-10-10T21:28:07.176Z,NaN,2025-11-03T18:45:58.758Z,NaN,NaN,NaN
97,flare-networks,flr,Flare,https://coin-images.coingecko.com/coins/images...,0.014817,1162818127,98,1551909587,1.262668e+07,0.016169,...,-90.11659,2023-01-10T03:14:05.921Z,0.008274,7.926322e+01,2023-10-19T03:35:41.663Z,NaN,2025-11-03T18:45:58.657Z,NaN,NaN,NaN
98,renzo-restaked-eth,ezeth,Renzo Restaked ETH,https://coin-images.coingecko.com/coins/images...,3872.990000,1157957372,99,1157957372,2.012185e+06,4157.100000,...,-25.90506,2025-08-24T19:25:30.766Z,1454.430000,1.662890e+02,2025-04-09T01:30:57.856Z,NaN,2025-11-03T18:45:58.625Z,NaN,NaN,NaN


In [ ]:
# Définir une fonction d'éxtraction
def extract_data_from_api():

    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        'vs_currency': 'usd',
        'order': 'market_cap_desc',
        'per_page': 1000, # Obtenir les 1000 premières cryptomonnaies de la capitalisation
        'page': 1,
        'sparline': 'false'
    }

    # Requéte de l'API
    response = requests.get(url, params = params)

    # Verifier le statut si le code fonctionne
    if response.status_code == 200: # C'est à dire que la requéte à fonctionner
        data = response.json()
        return pd.json_normalize(data) # Convertir le json en Dataframe
    else:
        raise Exception("Erreur lors de l'extraction des données de l'API")


In [ ]:
# Extraction des données et afficher les 5 premières lignes
df_crypto = extract_data_from_api()
df_crypto.head()

,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated,roi.times,roi.currency,roi.percentage
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,107306.00,2141904317375,1,2141906357902,6.778497e+10,110665.00,...,-14.88129,2025-10-06T18:57:42.558Z,67.810000,1.581645e+05,2013-07-06T00:00:00.000Z,NaN,2025-11-03T18:29:00.637Z,NaN,NaN,NaN
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,3656.47,441921463369,2,441921463369,4.340939e+10,3911.96,...,-25.95617,2025-08-24T19:21:03.333Z,0.432979,8.457251e+05,2015-10-20T00:00:00.000Z,NaN,2025-11-03T18:29:00.539Z,44.567528,btc,4456.752835
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,1.00,183463761355,3,183463761355,1.297533e+11,1.00,...,-24.41861,2018-07-24T00:00:00.000Z,0.572521,7.466849e+01,2015-03-02T00:00:00.000Z,NaN,2025-11-03T18:29:03.518Z,NaN,NaN,NaN
3,ripple,xrp,XRP,https://coin-images.coingecko.com/coins/images...,2.37,142363210750,4,236815157165,5.017362e+09,2.53,...,-34.93275,2025-07-18T03:40:53.808Z,0.002686,8.822521e+04,2014-05-22T00:00:00.000Z,NaN,2025-11-03T18:29:03.115Z,NaN,NaN,NaN
4,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,1001.16,138122911508,5,138122911137,3.561920e+09,1086.67,...,-26.73344,2025-10-13T08:41:24.131Z,0.039818,2.520753e+06,2017-10-19T00:00:00.000Z,NaN,2025-11-03T18:29:00.546Z,NaN,NaN,NaN


In [23]:
# La dimension
df_crypto.shape

(100, 29)

In [17]:
# L'ensemble des information sur la base
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 29 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                100 non-null    object 
 1   symbol                            100 non-null    object 
 2   name                              100 non-null    object 
 3   image                             100 non-null    object 
 4   current_price                     100 non-null    float64
 5   market_cap                        100 non-null    int64  
 6   market_cap_rank                   100 non-null    int64  
 7   fully_diluted_valuation           100 non-null    int64  
 8   total_volume                      100 non-null    float64
 9   high_24h                          100 non-null    float64
 10  low_24h                           100 non-null    float64
 11  price_change_24h                  100 non-null    float64
 12  price_cha

In [35]:
# Transformation des données
def transform_data(df):
    
    # Séléctionner les colonnes que je souhaite dans ma base
    df_transformed = df[['id', 'symbol', 'name', 'current_price', 'market_cap', 'total_volume', 'price_change_percentage_24h', 'last_updated']].copy()

    # Renommer les colonnes
    df_transformed.columns = ['id', 'symbol', 'name', 'price_usd', 'market_cap_usd', 'volume_24h_usd', 'price_change_percentage_24h', 'date']

    # Convertir la colonne 'date' (anciennement 'last_update') au format datetime sans l'heure
    df_transformed['date'] = pd.to_datetime(df_transformed['date']).dt.date

    # Remplacer les valeurs manquants par 0
    df_transformed = df_transformed.fillna(0)
    return df_transformed

In [36]:
# Appliquer la transformation sur la base de données
# Transformer les données 
df_transformed = transform_data(df_crypto)
df_transformed.head()

,id,symbol,name,price_usd,market_cap_usd,volume_24h_usd,price_change_percentage_24h,date
0,bitcoin,btc,Bitcoin,107306.00,2141904317375,6.778497e+10,-2.64393,2025-11-03
1,ethereum,eth,Ethereum,3656.47,441921463369,4.340939e+10,-5.26403,2025-11-03
2,tether,usdt,Tether,1.00,183463761355,1.297533e+11,0.00983,2025-11-03
3,ripple,xrp,XRP,2.37,142363210750,5.017362e+09,-5.13766,2025-11-03
4,binancecoin,bnb,BNB,1001.16,138122911508,3.561920e+09,-7.13026,2025-11-03


In [ ]:
# Stocker sous csv
#df_transformed.to_csv("analyse.csv")

In [59]:
# Chargement

import sqlite3

# Fonction pour charger les donner dans une base SQLite
def load_data_to_sqlite(df, db_file, table_name):
    # Connexion à la base de données SQLite (ou création si elle n'existe)
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    
    # Créer la table si elle n'existe pas déja
    cursor.execute(f'''
                   CREATE TABLE IF NOT EXISTS {table_name} (
                   id TEXT PRIMARY KEY,
                   symbol TEXT,
                   name TEXT,
                   price_usd REAL,
                   market_cap_usd REAL,
                   volume_24h_usd REAL,
                   price_change_24h_percent REAL
                )
            ''')
    
    # Charger les données dans la table
    df.to_sql(table_name, conn, if_exists='replace', index=False)
        
    # Valider et fermer la connexion
    conn.commit()
    conn.close()
    print(f"Les données ont été chargé dans la table '{table_name}' de la base {db_file}")
    
# Exemple de fichier de base de données et tabke
db_file = 'Données_database.db'
table_name = 'crypto_data'

# Charger les données dans SQLite
load_data_to_sqlite(df_transformed, db_file, table_name)

Les données ont été chargé dans la table 'crypto_data' de la base Données_database.db


In [61]:
# Pipeline ETL complet
def etl_pipeline():
    
    # Extraction
    df_crypto = extract_data_from_api()
    print("Extraction réussi")
    
    # Chargement
    db_file = 'Données_database.db'
    table_name = 'crypto_data'
    load_data_to_sqlite(df_transformed, db_file, table_name)
    print("Chargement réussi")
    
# Exécuter le pipeline ETL
etl_pipeline() 

Extraction réussi
Les données ont été chargé dans la table 'crypto_data' de la base Données_database.db
Chargement réussi


In [ ]:
# Vérification du contenu de la base
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

# Lister toutes les tables présentes
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables présentes dans la base :", tables)

conn.close()

Tables présentes dans la base : [('crypto_data',)]


In [62]:
# Afficher les premières lignes de la table pour être sûr que les données ont bien été chargées :
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

cursor.execute("SELECT * FROM crypto_data LIMIT 5;")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

('bitcoin', 'btc', 'Bitcoin', 107306.0, 2141904317375, 67784973009.0, -2.64393, '2025-11-03')
('ethereum', 'eth', 'Ethereum', 3656.47, 441921463369, 43409391320.0, -5.26403, '2025-11-03')
('tether', 'usdt', 'Tether', 1.0, 183463761355, 129753264616.0, 0.00983, '2025-11-03')
('ripple', 'xrp', 'XRP', 2.37, 142363210750, 5017361647.0, -5.13766, '2025-11-03')
('binancecoin', 'bnb', 'BNB', 1001.16, 138122911508, 3561920434.0, -7.13026, '2025-11-03')


Utilisation de pandas pour importer les fichiers parquets

In [4]:
file_path = "histo_data_files/yellow_tripdata_2025-01.parquet"
df = pd.read_parquet(file_path)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [9]:
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [ ]:
# Convertir la colonne 'store_and_fwd_flag' en type numerique en forçant les erreurs à NAN
df['store_and_fwd_flag'] = pd.to_numeric(df['store_and_fwd_flag'], errors='coerce')


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     float64       
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

Comparer les performances en temps d’exécution de Pandas, Polars et PySpark

Pandas

In [16]:
%%timeit
# Groupement et aggrégation
df_agg = df.groupby(['tpep_pickup_datetime', 'VendorID'])[['fare_amount', 'extra']].agg(["mean", "sum", "max"])
# Afficher le résultat
df_agg = df_agg.reset_index()
df_agg
df_agg.to_parquet("temp_pandas.parquet")
pd.read_parquet("temp_pandas.parquet").head(30)

3.4 s ± 149 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Polars

In [19]:
%%timeit
df_polars = (
    pl.scan_parquet(file_path)
    # Convertir 'store_and_fwd_flag' en float
    .with_columns([pl.col("store_and_fwd_flag").cast(pl.Float64, strict=False)])
    .group_by(["tpep_pickup_datetime", "VendorID"])
    .agg(
        [
            pl.col("PULocationID").mean().alias("avg_PULO"),
            pl.col("PULocationID").sum().alias("sum_PULO"),
            pl.col("PULocationID").max().alias("max_PULO"),
            
        ]
    )
)

# Collécter le résultat
df_resultat = df_polars.collect()
# Sauvegarder le resultat au format parquet
df_resultat.write_parquet("temp_polars.parquet")


1.4 s ± 76.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Pyspark

In [8]:
# Import SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, max, sum, col


# Création SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("airline_example") \
    .getOrCreate()

print("✅ Spark lancé avec succès — version :", spark.version)


✅ Spark lancé avec succès — version : 4.0.1


In [9]:

file_path = "histo_data_files/yellow_tripdata_2025-01.parquet"

# Lecture du fichier Parquet
df_spark = spark.read.parquet(file_path)
df_spark.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [10]:
# Aperçu et structure
df_spark.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [11]:

# Convertir la colonne 'store_and_fwd' en numerique
df_spark = df_spark.withColumn("store_and_fwd_flag", col("store_and_fwd_flag").cast("float"))


In [12]:
# Aperçu et structure
df_spark.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: float (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [21]:

# Quelques statistiques de base
df_spark.select(
    avg(col("trip_distance")).alias("distance_moyenne"),
    avg(col("fare_amount")).alias("tarif_moyen"),
    max(col("fare_amount")).alias("tarif_max"),
    min(col("fare_amount")).alias("tarif_min"),
    count("*").alias("nb_courses")
).show()

+-----------------+-----------------+---------+---------+----------+
| distance_moyenne|      tarif_moyen|tarif_max|tarif_min|nb_courses|
+-----------------+-----------------+---------+---------+----------+
|5.855126178843539|17.08180276045484|863372.12|   -900.0|   3475226|
+-----------------+-----------------+---------+---------+----------+



In [20]:
%%timeit
file_path = "histo_data_files/yellow_tripdata_2025-01.parquet"

# Lecture du fichier Parquet
df_spark = spark.read.parquet(file_path)

84.6 ms ± 11.9 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
